# 04 ? Mode Choice Join Preview and Analytical Dataset

This notebook first builds a local, stratified 350-trip preview to inspect the relationships among `viajes`, `personas`, `hogares`, vehicles, stages, and the ZAT distance table. It then uses the investigator-supplied classification to construct a full analytical dataset with one row per eligible trip.

The preview is exploratory and not representative. The full dataset retains source values, keeps both observed mode fields, and does not apply the expansion factor. Source records are never modified. Vehicle summaries are household-level attributes repeated on trip rows; they do not establish vehicle availability to a particular person or trip. Stage summaries remain preview-only.


In [ ]:
from pathlib import Path
import pandas as pd
from mode_choice.mode_choice_join_preview import build_join_preview

PROJECT_ROOT = next((candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (candidate / 'pyproject.toml').is_file()), None)
assert PROJECT_ROOT is not None, 'Could not locate the project root.'
preview_path, summary = build_join_preview(PROJECT_ROOT, sample_size=350, random_seed=5082026)
preview = pd.read_csv(preview_path, dtype=str, keep_default_na=False)
assert len(preview) == summary['preview_rows'] == 350
assert preview['key_viaje'].nunique() == len(preview)
assert preview['persona__key_persona'].ne('').all()
assert preview['hogar__key_hg'].ne('').all()
assert sum(summary['sample_mode_counts'].values()) == 350
assert summary['distance_table_validation'] == {'row_count': 1475010, 'unique_directed_pairs': 1475010, 'zone_count': 1215}
assert summary['distance_status_counts'].get('unmatched_zone_pair', 0) == 0
assert summary['distance_status_counts'].get('missing_zone_code', 0) == 0
assert (preview.loc[preview.zat_distance_status == 'intrazonal_zero_derived', 'zat_centroid_distance_m'] == '0').all()
summary

In [ ]:
from IPython.display import display

preview_path = PROJECT_ROOT / 'data/interim/mode_choice_trip_join_preview_350.csv'
preview = pd.read_csv(preview_path, low_memory=False)
assert len(preview) == 350
print(f"Initial preview: {len(preview.head(10))} of {len(preview)} rows; {len(preview.columns)} columns")
display(preview.head(10))
column_profile = pd.DataFrame({
    'column': preview.columns,
    'inferred_type': [str(preview[column].dtype) for column in preview.columns],
    'missing_values': [int(preview[column].isna().sum()) for column in preview.columns],
    'distinct_values': [int(preview[column].nunique(dropna=True)) for column in preview.columns],
})
assert len(column_profile) == len(preview.columns) == 127
display(column_profile)


## Preview field descriptions and observed duplication

Exact copies, high match rates, and differences are reported separately. The 95% threshold is a practical flag only; it is not a statistical significance test and does not by itself justify dropping a field.


In [ ]:
field_dictionary_path = PROJECT_ROOT / 'docs/data_dictionary/mode_choice_trip_dataset_fields.csv'
field_dictionary = pd.read_csv(field_dictionary_path, dtype=str, keep_default_na=False)
field_dictionary = field_dictionary.set_index(['source_table', 'source_field'])

known_descriptions = {
    'key_hg': 'Technical household identifier.', 'key_pers': 'Person identifier linked to the trip.',
    'key_pers_viaja': 'Alternate identifier for the traveling person.', 'key_viaje': 'Unique technical trip identifier.',
    'zat_ori': 'Trip origin zone code.', 'zat_des': 'Trip destination zone code.',
    'zat_hg': 'Zone code associated with the household residence.',
    'modo_principal_agrupado': 'Observed grouped main travel mode.',
    'modo_principal_desagrupado': 'Observed detailed main travel mode.', 'fexp_vj': 'Trip expansion factor.',
}

def describe_preview_column(column):
    if column.startswith('persona__'):
        level, raw_name, table = 'person', column.removeprefix('persona__'), 'personas'
    elif column.startswith('hogar__'):
        level, raw_name, table = 'household', column.removeprefix('hogar__'), 'hogares'
    else:
        level, raw_name, table = 'trip', column, 'viajes'
    documented = ''
    if (table, raw_name) in field_dictionary.index:
        documented = field_dictionary.loc[(table, raw_name), 'documented_definition']
    if documented:
        return level, documented, 'Documented in local dictionary'
    if raw_name in known_descriptions:
        return level, known_descriptions[raw_name], 'Functional description; verify against survey documentation'
    return level, f"Field '{raw_name.replace('_', ' ')}' (description inferred from field name; confirm with survey documentation).", 'Inferred; official definition pending'

columns = list(preview.columns)
zat_columns = [column for column in ['zat_hg', 'zat_ori', 'zat_des', 'persona__zat_hg', 'hogar__zat_hg'] if column in preview]
zat_comparisons = []
for i, left in enumerate(zat_columns):
    for right in zat_columns[i + 1:]:
        comparable = preview[left].notna() & preview[right].notna() & preview[left].astype(str).str.strip().ne('') & preview[right].astype(str).str.strip().ne('')
        count = int(comparable.sum())
        matches = int(preview.loc[comparable, left].astype(str).eq(preview.loc[comparable, right].astype(str)).sum())
        zat_comparisons.append({'pair': f'{left} ? {right}', 'comparable_rows': count, 'matching_rows': matches,
                                'match_percent': round(100 * matches / count, 1) if count else None})

field_rows = []
for column in columns:
    level, description, definition_status = describe_preview_column(column)
    field_rows.append({'column': column, 'level/source': level, 'description_en': description,
                       'definition_status': definition_status,
                       'possible_repetition': 'See pairwise exact-copy and ZAT agreement checks below.'})
field_description_table = pd.DataFrame(field_rows)
assert len(field_description_table) == 127
display(field_description_table)
display(pd.DataFrame(zat_comparisons))
print('ZAT agreement is descriptive and does not imply conceptual equivalence. Percentages use comparable nonmissing rows only.')


## Relational structure

Keep one row per `key_viaje`. Link trips to people with `key_pers = key_persona` and to households with `key_hg`. Aggregate one-to-many vehicle records to household level before joining. Do not join raw stage rows directly to trips. Keep both observed mode fields as targets; they do not provide attributes for unchosen alternatives.


## Full analytical dataset

The next cells use the investigator-provided classification and the existing read-only relational database. The original 47-column trip dataset is preserved. The new output retains every eligible trip, both observed mode outcomes, selected predictors and identifiers, and unresolved duplicate candidates. It does not select a final model specification, impute values, recode categories, apply weights, or create time-of-day bands. Build those bands later in the EDA from a histogram.


In [ ]:
import re
import sqlite3
from io import StringIO
from mode_choice.mode_choice_join_preview import _canonical_zone, _read_distance_table

classification_path = PROJECT_ROOT / 'docs/mode_choice_variable_classification.md'
classification_document = classification_path.read_text(encoding='utf-8')
classification_tsv = re.search(r'```tsv\s*\n(.*?)\n```', classification_document, flags=re.S)
assert classification_tsv is not None, 'The investigator-supplied TSV classification is missing.'
classification = pd.read_csv(StringIO(classification_tsv.group(1)), sep='\t', dtype=str, keep_default_na=False)
classification['COLUMNA'] = classification['COLUMNA'].str.strip()
assert classification.shape == (126, 4) and classification['COLUMNA'].is_unique
assert classification['Decision Base Inicial'].value_counts().to_dict() == {'NO': 66, 'Pending_duplicate_validation': 25, 'SI': 35}

database_path = PROJECT_ROOT / 'data/relational/em2023_relational.sqlite'
connection = sqlite3.connect(f'file:{database_path.as_posix()}?mode=ro', uri=True)
try:
    trips = pd.read_sql_query('SELECT * FROM "viajes" ORDER BY rowid', connection)
    people = pd.read_sql_query('SELECT * FROM "personas"', connection)
    households = pd.read_sql_query('SELECT * FROM "hogares"', connection)
    vehicles = pd.read_sql_query('SELECT * FROM "vehiculos"', connection)
    stages = pd.read_sql_query('SELECT * FROM "etapas"', connection)
    source_schemas = {table: {row[1]: row[2] for row in connection.execute(f'PRAGMA table_info("{table}")')} for table in ['viajes','personas','hogares','vehiculos','etapas']}
finally:
    connection.close()

for source_frame in (trips, people, households, vehicles, stages):
    source_frame.columns = source_frame.columns.map(lambda name: name.strip() if isinstance(name, str) else name)

source_trip_rows = len(trips)
assert source_trip_rows == 100174
assert trips['key_viaje'].notna().all() and trips['key_viaje'].is_unique
assert people['key_persona'].notna().all() and people['key_persona'].is_unique
assert households['key_hg'].notna().all() and households['key_hg'].is_unique

people_join = people.add_prefix('persona__')
households_join = households.add_prefix('hogar__')
full = trips.merge(people_join, left_on='key_pers', right_on='persona__key_persona', how='left', validate='many_to_one', indicator='_person_link')
unmatched_person_rows = int(full['_person_link'].eq('left_only').sum())
full = full.drop(columns='_person_link')
full = full.merge(households_join, left_on='key_hg', right_on='hogar__key_hg', how='left', validate='many_to_one', indicator='_household_link')
unmatched_household_rows = int(full['_household_link'].eq('left_only').sum())
full = full.drop(columns='_household_link')
assert unmatched_person_rows == 0 and unmatched_household_rows == 0
assert len(full) == source_trip_rows and full['key_viaje'].is_unique

vehicle_summary = (vehicles.groupby('key_hg', as_index=False)
    .agg(vehicle_records_in_household=('key_hg','size'),
         vehicle_type_values=(next(column for column in vehicles.columns if column.startswith('tipo_veh')),lambda values: ' | '.join(values.dropna().astype(str)))))
full = full.merge(vehicle_summary,on='key_hg',how='left',validate='many_to_one')
full['vehicle_records_in_household'] = full['vehicle_records_in_household'].fillna(0).astype('int64')
vehicles['cantidad_numeric'] = pd.to_numeric(vehicles['cantidad'], errors='coerce')
vehicle_quantity = vehicles.groupby('key_hg',as_index=False)['cantidad_numeric'].sum().rename(columns={'cantidad_numeric':'vehicle_quantity_sum_exploratory'})
full = full.merge(vehicle_quantity,on='key_hg',how='left',validate='many_to_one')
full['vehicle_quantity_sum_exploratory'] = full['vehicle_quantity_sum_exploratory'].fillna(0)

stage_summary = stages.groupby('key_viaje',as_index=False).agg(
    stage_count=('key_etapa','size'),
    stage_mode_sequence=('modo_etapa',lambda values:' | '.join(values.dropna().astype(str))),
    stage_access_minutes_sum_exploratory=('t_acceso_min',lambda values:pd.to_numeric(values,errors='coerce').sum(min_count=1)),
    stage_wait_minutes_sum_exploratory=('t_espera_min',lambda values:pd.to_numeric(values,errors='coerce').sum(min_count=1)),
    stage_egress_minutes_sum_exploratory=('t_egreso_min',lambda values:pd.to_numeric(values,errors='coerce').sum(min_count=1)),
    stage_cost_sum_exploratory=('costo_viaje',lambda values:pd.to_numeric(values,errors='coerce').sum(min_count=1)),
    stage_payment_sum_exploratory=('pago_viaje',lambda values:pd.to_numeric(values,errors='coerce').sum(min_count=1)),
    stage_parking_cost_sum_exploratory=('costo_parqueo',lambda values:pd.to_numeric(values,errors='coerce').sum(min_count=1)))
full = full.merge(stage_summary,on='key_viaje',how='left',validate='one_to_one')
full['stage_count']=full['stage_count'].fillna(0).astype('int64')
full['vehicle_records_in_household'] = full['vehicle_records_in_household'].fillna(0).astype('int64')
classified_available = set(classification['COLUMNA']).issubset(set(full.columns) | {'zat_centroid_distance_m'})
assert classified_available, f"Classified fields missing: {sorted(set(classification['COLUMNA'])-set(full.columns)-{'zat_centroid_distance_m'})}"
print({'source_trip_rows':source_trip_rows,'joined_rows':len(full),'person_unmatched_rows':unmatched_person_rows,
       'household_unmatched_rows':unmatched_household_rows,'classified_fields':len(classification),
       'preview_only_unclassified_field':'zat_distance_status'})


In [ ]:
pending_fields = classification.loc[classification['Decision Base Inicial'].eq('Pending_duplicate_validation'),'COLUMNA'].tolist()
selected_fields = classification.loc[classification['Decision Base Inicial'].eq('SI'),'COLUMNA'].tolist()
explicit_counterparts = {'persona__cod_per':'cod_pers','persona__cod_hg':'cod_hg','persona__key_hg':'key_hg',
                         'persona__key_persona':'key_pers','hogar__cod_hog':'cod_hg'}
pending_comparisons=[]
for field in pending_fields:
    if field in explicit_counterparts:
        counterpart=explicit_counterparts[field]
    elif field.startswith('persona__'):
        candidate=field.removeprefix('persona__'); counterpart=candidate if candidate in trips.columns else None
    elif field.startswith('hogar__'):
        candidate=field.removeprefix('hogar__'); counterpart=candidate if candidate in trips.columns else None
    else: counterpart=None
    if counterpart is None or counterpart not in full.columns:
        pending_comparisons.append({'pending_field':field,'counterpart_field':'','comparison_level':'',
          'linked_trip_rows':len(full),'comparable_rows':0,'equal_rows':0,'match_percent':None,
          'different_nonmissing_rows':0,'difference_percent':None,'pending_missing_rows':int(full[field].isna().sum()),
          'counterpart_missing_rows':None,'pending_only_missing_rows':None,'counterpart_only_missing_rows':None,
          'both_missing_rows':None,'coverage_percent':0.0,'possible_duplicate_relevant':False,'exact_value_copy':False,
          'decision':'retain as pending; no directly comparable counterpart identified'})
        continue
    left,right=full[field],full[counterpart]
    left_missing=left.isna()|left.astype('string').str.strip().eq('')
    right_missing=right.isna()|right.astype('string').str.strip().eq('')
    comparable=~left_missing&~right_missing
    equal=left.loc[comparable].astype(str).eq(right.loc[comparable].astype(str))
    n=int(comparable.sum()); matches=int(equal.sum()); differences=n-matches
    rate=100*matches/n if n else None
    left_only=int((left_missing&~right_missing).sum()); right_only=int((right_missing&~left_missing).sum())
    both_missing=int((left_missing&right_missing).sum()); coverage=100*n/len(full) if len(full) else 0.0
    possible=rate is not None and rate>=95
    exact=n>0 and differences==0 and left_only==0 and right_only==0
    if exact and counterpart in selected_fields: decision='exclude pending copy; retain the selected canonical field'
    elif possible: decision='retain pending field; high agreement is not exact and needs review'
    else: decision='retain pending field; values differ or do not support a duplicate finding'
    pending_comparisons.append({'pending_field':field,'counterpart_field':counterpart,
      'comparison_level':'trip-person' if field.startswith('persona__') else 'trip-household',
      'linked_trip_rows':len(full),'comparable_rows':n,'equal_rows':matches,
      'match_percent':round(rate,3) if rate is not None else None,'different_nonmissing_rows':differences,
      'difference_percent':round(100*differences/n,3) if n else None,
      'pending_missing_rows':int(left_missing.sum()),'counterpart_missing_rows':int(right_missing.sum()),
      'pending_only_missing_rows':left_only,'counterpart_only_missing_rows':right_only,
      'both_missing_rows':both_missing,'coverage_percent':round(coverage,3),
      'possible_duplicate_relevant':possible,'exact_value_copy':exact,'decision':decision})
duplicate_validation=pd.DataFrame(pending_comparisons)
assert len(duplicate_validation)==25
display(duplicate_validation)
print({'pending_fields':len(pending_fields),'possible_duplicate_relevant_at_95_percent':int(duplicate_validation['possible_duplicate_relevant'].sum()),
       'fields_without_direct_counterpart':int(duplicate_validation['counterpart_field'].eq('').sum())})


In [ ]:
import hashlib
import os
import tempfile
import numpy as np

# Convert source decimal-hour values without modifying their original columns.
def decimal_hour_minutes(series):
    values = pd.to_numeric(series.astype('string').str.strip().str.replace(',', '.', regex=False), errors='coerce')
    return np.rint(values * 60).astype('Int64')

start_minutes = decimal_hour_minutes(full['hora_ini'])
end_minutes = decimal_hour_minutes(full['hora_fin'])
reported_duration = pd.to_numeric(full['duracion_min'].astype('string').str.strip().str.replace(',', '.', regex=False), errors='coerce')
full['start_minutes_after_midnight'] = start_minutes
full['end_minutes_after_midnight'] = end_minutes
full['start_time_interpreted'] = start_minutes.map(lambda value: f'{int(value)//60:02d}:{int(value)%60:02d}' if pd.notna(value) else pd.NA)
full['end_time_interpreted'] = end_minutes.map(lambda value: f'{int(value)//60:02d}:{int(value)%60:02d}' if pd.notna(value) else pd.NA)
full['duration_calculated_min'] = ((end_minutes - start_minutes) % 1440).astype('Int64')
full['crosses_midnight'] = (end_minutes < start_minutes).where(start_minutes.notna() & end_minutes.notna(), pd.NA).astype('boolean')

origin = full['zat_ori'].map(lambda value: _canonical_zone(value, 'zat_ori'))
destination = full['zat_des'].map(lambda value: _canonical_zone(value, 'zat_des'))
needed_pairs = {(int(a), int(b)) for a, b in zip(origin, destination) if a is not None and b is not None and a != b}
distance_path = PROJECT_ROOT / 'data/sources/raw/zonification/ZAT_Distance.csv'
distance_lookup, distance_validation = _read_distance_table(distance_path, needed_pairs)
full['zat_centroid_distance_m'] = [0 if a is not None and b is not None and a == b else distance_lookup.get((a,b), pd.NA) if a is not None and b is not None else pd.NA for a,b in zip(origin,destination)]

selected_fields = classification.loc[classification['Decision Base Inicial'].eq('SI'),'COLUMNA'].tolist()
exact_copy_fields = set(duplicate_validation.loc[duplicate_validation['exact_value_copy'] & duplicate_validation['counterpart_field'].isin(selected_fields),'pending_field'])
pending_retained = [field for field in pending_fields if field not in exact_copy_fields]
source_columns = list(dict.fromkeys(selected_fields + pending_retained))
derived_columns = ['zat_centroid_distance_m','start_time_interpreted','end_time_interpreted','start_minutes_after_midnight','end_minutes_after_midnight','duration_calculated_min','crosses_midnight','vehicle_records_in_household','vehicle_type_values']
output_columns = list(dict.fromkeys(source_columns + derived_columns))
assert 'zat_distance_status' not in output_columns
analytical = full.loc[:, output_columns].copy()
assert len(analytical) == source_trip_rows and analytical['key_viaje'].notna().all() and analytical['key_viaje'].is_unique
assert {'modo_principal_agrupado','modo_principal_desagrupado','key_viaje','fexp_vj'}.issubset(analytical.columns)



In [ ]:
from pathlib import Path
import hashlib
import tempfile

# Safe, repeatable artifact writes: existing identical files are accepted; changed files are never overwritten.
def write_csv_idempotent(frame, target):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8-sig', newline='', suffix='.csv', dir=target.parent, delete=False) as stream:
        temp_path = Path(stream.name)
        frame.to_csv(stream, index=False)
    if target.exists():
        existing_hash = hashlib.sha256(target.read_bytes()).hexdigest()
        candidate_hash = hashlib.sha256(temp_path.read_bytes()).hexdigest()
        temp_path.unlink()
        if existing_hash != candidate_hash:
            raise FileExistsError(f'Refusing to overwrite an existing artifact with different content: {target}')
        return 'unchanged'
    os.replace(temp_path, target)
    return 'created'

# All pairwise comparisons and pending decisions are retained in the validation matrix.
duplicate_output = PROJECT_ROOT / 'outputs/tables/mode_choice_duplicate_validation.csv'
duplicate_write_status = write_csv_idempotent(duplicate_validation, duplicate_output)


# Construct a field-level dictionary that matches the exact exported schema.
derived_definitions = {
 'zat_centroid_distance_m': ('Distance lookup','trip OD pair','float64','predictor','Included; derived','Distance between ZAT centroids in meters; intrazonal pairs are zero.'),
 'start_time_interpreted': ('viajes','trip','string','derived time','Included; derived','Decimal-hour source converted to HH:MM; source value retained.'),
 'end_time_interpreted': ('viajes','trip','string','derived time','Included; derived','Decimal-hour source converted to HH:MM; source value retained.'),
 'start_minutes_after_midnight': ('viajes','trip','Int64','derived time','Included; derived','Rounded decimal hours multiplied by 60.'),
 'end_minutes_after_midnight': ('viajes','trip','Int64','derived time','Included; derived','Rounded decimal hours multiplied by 60.'),
 'duration_calculated_min': ('viajes','trip','Int64','derived time','Included; derived','End minus start modulo 24 hours.'),
 'crosses_midnight': ('viajes','trip','boolean','quality/time','Included; derived','True when end time is earlier than start time.'),
 'vehicle_records_in_household': ('vehiculos','household','int64','household context','Included; aggregate','Count of vehicle records for the household; not person-level availability.'),
 'vehicle_type_values': ('vehiculos','household','object','household context','Included; aggregate','Distinct source records concatenated; not person-level availability.'),
}
classification_by_field = classification.set_index('COLUMNA').to_dict('index')
field_dictionary_rows=[]
for column in analytical.columns:
    if column in derived_definitions:
        table, level, dtype, role, decision, transformation = derived_definitions[column]
        source_field = ''
        limitation = 'Derived value; see transformation and QA report.'
    else:
        row=classification_by_field[column]
        role=row['Possible research_role']
        decision=row['Decision Base Inicial']
        table='personas' if column.startswith('persona__') else 'hogares' if column.startswith('hogar__') else 'viajes'
        source_field=column.removeprefix('persona__').removeprefix('hogar__')
        level='person' if table=='personas' else 'household' if table=='hogares' else 'trip'
        dtype=str(analytical[column].dtype)
        transformation='Joined by validated technical key; source value retained.' if level!='trip' else 'Source value retained.'
        limitation='Pending duplicate validation; not approved as a model predictor.' if decision=='Pending_duplicate_validation' else ''
        source_dict=PROJECT_ROOT/'docs/data_dictionary/mode_choice_trip_dataset_fields.csv'
        documented=pd.read_csv(source_dict,dtype=str,keep_default_na=False)
        match=documented[(documented['source_table']==table)&(documented['source_field']==source_field)]
        description=match.iloc[0]['documented_definition'] if len(match) and match.iloc[0]['documented_definition'] else f"Description inferred from source field name: {source_field}."
        if not limitation: limitation='Official definition unavailable; verify against survey documentation.' if description.startswith('Description inferred') else ''
    if column in derived_definitions:
        description=transformation
    field_dictionary_rows.append({'column':column,'source_table':table,'source_field':source_field,'level':level,'data_type':dtype,
      'research_role':role,'inclusion_decision':decision,'description_en':description,'transformation':transformation,
      'missing_values':int(analytical[column].isna().sum()),'limitations':limitation})
field_dictionary_export=pd.DataFrame(field_dictionary_rows)
assert field_dictionary_export['column'].tolist()==analytical.columns.tolist()

# Aggregate QA indicators; no trip/person identifiers are written to this report.
complete_times=start_minutes.notna() & end_minutes.notna() & reported_duration.notna()
duration_difference=(full.loc[complete_times,'duration_calculated_min'].astype(float)-reported_duration.loc[complete_times]).abs()
quality_rows=[
 {'metric':'source_trip_rows','value':source_trip_rows,'detail':'All eligible viajes records'},
 {'metric':'analytical_rows','value':len(analytical),'detail':'No missingness-based exclusions'},
 {'metric':'analytical_columns','value':len(analytical.columns),'detail':'Matches the field dictionary'},
 {'metric':'key_viaje_unique_nonnull','value':bool(analytical['key_viaje'].notna().all() and analytical['key_viaje'].is_unique),'detail':'One row per trip'},
 {'metric':'person_unmatched_rows','value':unmatched_person_rows,'detail':'Validated many-to-one join'},
 {'metric':'household_unmatched_rows','value':unmatched_household_rows,'detail':'Validated many-to-one join'},
 {'metric':'pending_fields_tested','value':len(duplicate_validation),'detail':'All pending fields represented in duplicate matrix'},
 {'metric':'pending_exact_copies_consolidated','value':len(exact_copy_fields),'detail':'Only exact copies of selected fields excluded'},
 {'metric':'complete_time_records','value':int(complete_times.sum()),'detail':'Start, end, reported duration available'},
 {'metric':'time_records_with_any_missing_component','value':int((~complete_times).sum()),'detail':'Retained in analytical dataset'},
 {'metric':'duration_exact_matches','value':int((duration_difference==0).sum()),'detail':'Calculated versus reported, complete records'},
 {'metric':'duration_matches_within_one_minute','value':int((duration_difference<=1).sum()),'detail':'Calculated versus reported, complete records'},
 {'metric':'duration_differences_over_one_minute','value':int((duration_difference>1).sum()),'detail':'Calculated versus reported, complete records'},
 {'metric':'trips_crossing_midnight','value':int(full['crosses_midnight'].fillna(False).sum()),'detail':'End time earlier than start time'},
 {'metric':'ZAT_distance_lookup_validation','value':str(distance_validation),'detail':'Distance table validation summary'},
 {'metric':'ZAT_distance_missing','value':int(analytical['zat_centroid_distance_m'].isna().sum()),'detail':'Missing OD zone or unmatched lookup'},
 {'metric':'expansion_factor_applied','value':False,'detail':'fexp_vj retained as source variable'},
 {'metric':'time_of_day_bands_created','value':False,'detail':'Bands deferred to EDA histogram'},
]
for column in analytical.columns:
    quality_rows.append({'metric':f'missing::{column}','value':int(analytical[column].isna().sum()),'detail':'Field-level missingness'})
for column in ['modo_principal_agrupado','modo_principal_desagrupado']:
    if column in analytical:
        counts=analytical[column].value_counts(dropna=False).to_dict()
        for category,count in counts.items(): quality_rows.append({'metric':f'distribution::{column}::{category}','value':int(count),'detail':'Observed category count'})
quality_report=pd.DataFrame(quality_rows)

data_output=PROJECT_ROOT/'data/analytical/em2023_mode_choice_analytical_dataset.csv'
dictionary_output=PROJECT_ROOT/'docs/data_dictionary/mode_choice_analytical_dataset_fields.csv'
quality_output=PROJECT_ROOT/'outputs/reports/mode_choice_analytical_dataset_quality.csv'
write_status={str(data_output):write_csv_idempotent(analytical,data_output),
               str(dictionary_output):write_csv_idempotent(field_dictionary_export,dictionary_output),
               str(quality_output):write_csv_idempotent(quality_report,quality_output)}
assert len(pd.read_csv(data_output,low_memory=False))==100174
assert len(pd.read_csv(dictionary_output))==len(analytical.columns)
assert len(pd.read_csv(duplicate_output))==25
print({'write_status':write_status,'rows':len(analytical),'columns':len(analytical.columns),
       'duplicate_candidates_at_95_percent':int(duplicate_validation['possible_duplicate_relevant'].sum()),
       'exact_copies_consolidated':len(exact_copy_fields),'duration_exact_matches':int((duration_difference==0).sum()),
       'duration_differences_over_one_minute':int((duration_difference>1).sum()),
       'crosses_midnight':int(full['crosses_midnight'].fillna(False).sum())})
